In [4]:
import pandas as pd

cruzbuy = pd.read_csv('./data/clean/cruzbuy_clean.csv')
cruzbuy.head(5)

,Transaction Date,Merchant Name,Item Description,Category,Subcategory,Item Name,Quantity,Subtotal,Total Price
0,2025-01-02,Complete Book & Media Supply Inc,Quanta And Fields: The Biggest Ideas In The Un...,Published Products,Printed Media,Nan,1.0,$17.68,$17.68
1,2025-01-02,New England Biolabs Inc,Nebridge® Ligase Master Mix,Nan,Nan,Nan,1.0,$55.80,$55.80
2,2025-01-02,New England Biolabs Inc,Nebuilder® Hifi Dna Assembly Master Mix,Nan,Nan,Nan,1.0,"$1,608.00","$1,608.00"
3,2025-01-02,New England Biolabs Inc,Nhei-Hf®,Nan,Nan,Nan,1.0,$46.80,$46.80
4,2025-01-02,New England Biolabs Inc,Noti-Hf®,Nan,Nan,Nan,1.0,$49.80,$49.80


In [6]:
import pandas as pd

# Load cleaned Amazon data
amazon_path = "./data/clean/amazon_clean.csv"
amazon_df = pd.read_csv(amazon_path)

# Parse date + money columns
amazon_df["Transaction Date"] = pd.to_datetime(amazon_df["Transaction Date"], errors="coerce")
amazon_df["Total Price"] = (
    amazon_df["Total Price"]
    .astype(str)
    .str.replace(r"[\$,]", "", regex=True)
    .str.strip()
)
amazon_df["Total Price"] = pd.to_numeric(amazon_df["Total Price"], errors="coerce")

# Drop bad rows
amazon_df = amazon_df.dropna(subset=["Transaction Date", "Total Price"])

# Monthly spend (YYYY-MM)
amazon_monthly_spend = (
    amazon_df
    .assign(Month=amazon_df["Transaction Date"].dt.strftime("%Y-%m"))
    .groupby("Month", as_index=False)["Total Price"]
    .sum()
    .rename(columns={"Total Price": "Amazon Spend"})
)

# Round for readability
amazon_monthly_spend["Amazon Spend"] = amazon_monthly_spend["Amazon Spend"].round(2)

amazon_monthly_spend


,Month,Amazon Spend
0,2025-01,346572.66
1,2025-02,359988.71
2,2025-03,276227.97
3,2025-04,606004.32
4,2025-05,771313.08
5,2025-06,468447.59
6,2025-07,184766.11
7,2025-08,246647.13
8,2025-09,228045.70


In [7]:
# Top 20 purchases by frequency and by total cost
import pandas as pd

dataset_path = './data/clean/cruzbuy_clean.csv'  # swap to amazon_clean.csv, onecard_clean.csv, or bookstore_clean.csv as needed
item_col = 'Item Description'
cost_col = 'Total Price'

df = pd.read_csv(dataset_path).copy()
df[item_col] = df[item_col].fillna('').astype(str).str.strip()
df = df[df[item_col] != '']

df[cost_col] = (
    df[cost_col]
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .str.strip()
)
df[cost_col] = pd.to_numeric(df[cost_col], errors='coerce').fillna(0.0)

top_20_by_frequency = (
    df.groupby(item_col)
    .size()
    .reset_index(name='Purchase Count')
    .sort_values('Purchase Count', ascending=False)
    .head(20)
)

top_20_by_cost = (
    df.groupby(item_col, as_index=False)[cost_col]
    .sum()
    .rename(columns={cost_col: 'Total Cost'})
    .sort_values('Total Cost', ascending=False)
    .head(20)
)

print('Top 20 Most Frequent Purchases')
display(top_20_by_frequency)

print('Top 20 Purchases by Total Cost')
display(top_20_by_cost)


Top 20 Most Frequent Purchases


,Item Description,Purchase Count
9511,Custom Dna Tube,602
26150,Shipping,268
13614,Freight,197
22570,Placeholder - Do Not Close,186
14518,Gratuity,170
8303,Co2 Dry Ice Nuggets,148
20926,Nitrogen Ind Liq 230Lt 22Psi,114
13645,Freight Estimate,104
12011,Ewaste Fees,83
31537,Universal Pipet Tips,61


Top 20 Purchases by Total Cost


,Item Description,Total Cost
3794,Acct#160710 Rachel Carson Dh-Bulk Food Purchase,1500000.00
3793,Acct#160672 College 9/10 Dh - Bulk Food Purchase,1499999.99
3757,Acct# 160678 Cowell Dh - Bulk Food Purchase,1499999.99
17911,Leica Stellaris Wll Confocal Microscope With S...,807280.17
3761,Acct# 160684 Crown Dh - Bulk Food Purchase,800000.00
27595,Standard Compute Nodes; Part #'S • S-As-2124Bt...,682458.00
7587,"Catalyst 9166I Ap (W6E, Tri-Band 4X4, Xor) W/R...",663628.39
12337,Facsdiscover S8 Brvyguv 5Laser,619250.00
7588,"Catalyst 9300 48-Port, 8Xmgig+40X5G 90W Upoe+,...",605836.16
19030,Merrill Market-Bulk Food Purchase,570000.00


In [9]:
# Find raw Amazon rows for specific gift-card product categories
import pandas as pd

amazon_raw_path = './data/raw/amazon.csv'
amazon_raw_df = pd.read_csv(amazon_raw_path).copy()

category_col = 'Amazon-Internal Product Category'
target_categories = [
    'Electronic Gift Card',
    'Gift Card',
    'Target Gift Card',
    'ACD Gift Card',
]

amazon_raw_df[category_col] = amazon_raw_df[category_col].fillna('').astype(str).str.strip()
gift_card_rows = amazon_raw_df[amazon_raw_df[category_col].isin(target_categories)].copy()

print(f'Found {len(gift_card_rows)} raw Amazon rows in the selected gift-card categories.')
display(gift_card_rows[[category_col]].value_counts().reset_index(name='Row Count'))

gift_card_rows.head(20)


Found 515 raw Amazon rows in the selected gift-card categories.


/var/folders/gh/q87xz5kn6tn4w9zcnvxk2jb00000gn/T/ipykernel_32967/2956149287.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  amazon_raw_df = pd.read_csv(amazon_raw_path).copy()


,Amazon-Internal Product Category,Row Count
0,Electronic Gift Card,459
1,Gift Card,33
2,Target Gift Card,14
3,ACD Gift Card,9


,Order Date,Order ID,Account Group,PO Number,Order Quantity,Currency,Order Subtotal,Order Shipping & Handling,Order Promotion,Order Tax,...,Department,Cost Center,Project Code,Location,Custom Field 1,Seller Name,Seller Credentials,Seller City,Seller State,Seller ZipCode
1,6/4/2025,111-0725963-0664218,UC Santa Cruz Pcard Group,NaN,18,USD,3600.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,09523-627610,Amazon.com,NaN,Seattle,WA,98109
4,6/26/2025,113-6665869-9925065,UC Santa Cruz Pcard Group,NaN,50,USD,2500.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
13,3/3/2025,111-3715751-1997006,UC Santa Cruz Pcard Group,NaN,7,USD,2565.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,20360-680230,Amazon.com,NaN,Seattle,WA,98109
17,1/9/2025,113-3471214-6591466,UC Santa Cruz Pcard Group,NaN,75,USD,1875.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
18,3/31/2025,113-2258327-1668241,UC Santa Cruz Pcard Group,NaN,75,USD,1875.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
25,4/9/2025,111-4407903-3235441,UC Santa Cruz Pcard Group,NaN,24,USD,1764.15,0.0,0.00,17.06,...,NaN,NaN,NaN,NaN,SlugCents Office Supplies,"Amazon Payments, Inc.",NaN,Seattle,WA,98109
30,6/25/2025,113-6903160-3804240,UC Santa Cruz Pcard Group,NaN,60,USD,2200.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,Amazon.com,NaN,Seattle,WA,98109
38,5/29/2025,111-5710924-2067414,UC Santa Cruz Pcard Group,NaN,43,USD,1752.71,0.0,-17.17,35.63,...,NaN,NaN,NaN,NaN,NaN,"Amazon Payments, Inc.",NaN,Seattle,WA,98109
41,3/4/2025,112-5492347-4762638,UC Santa Cruz Pcard Group,NaN,17,USD,1275.00,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,Room Showcase participants,Amazon.com,NaN,Seattle,WA,98109
43,4/16/2025,112-7375263-3101012,UC Santa Cruz Pcard Group,NaN,12,USD,1271.40,0.0,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,"Amazon Payments, Inc.",NaN,Seattle,WA,98109
